# Where Marketplaces Bleed Revenue
## A Behavioral Autopsy of User Disengagement

**Dataset:** REES46 e-commerce platform — October, November, December 2019  
**Scale:** 120M+ events | 9 columns | 3 months of full platform activity  
**Objective:** Diagnose where and when a marketplace loses users and revenue, and surface early warning signals a growth team can act on.

---

### Project Roadmap

| Chapter | Focus | Status |
|---------|-------|--------|
| 1 | Anatomy of the Funnel | 🔄 In progress |
| 2 | Where Users Die — The Abandonment Topology | ⏳ |
| 3 | Who Are the Disengagers? — User Anatomy | ⏳ |
| 4 | The Temporal Dimension | ⏳ |
| 5 | The Revenue Sensitivity Model | ⏳ |
| 6 | Early Warning Signals | ⏳ |
| — | Executive Summary | ⏳ |

In [18]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


---

## Chapter 1: Anatomy of the Funnel

The funnel is the skeleton of any disengagement autopsy. Before diagnosing *why* users leave, we need to establish *where* they leave — at three levels of resolution:

- **Event-level**: what fraction of raw interactions progress to the next stage?
- **User-level**: what fraction of unique users complete each transition?
- **Session-level**: within a single visit, at what moment does the user disengage?

Each level tells a different story. Collapsing them into one metric is the most common mistake in growth analytics.

---

### 1.1 Dataset Overview

In [8]:
structure = con.execute("""
    SELECT *
    FROM read_csv_auto('../data/raw/2019-Oct.csv')
    LIMIT 5
""").fetchdf()

print(structure.columns.tolist())
display(structure)

['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


### 1.2 Cross-Month Event Distribution

The dataset spans three months with fundamentally different commercial dynamics:

- **October**: organic baseline — users browsing with natural intent, no external price pressure
- **November**: Black Friday season — the single most distorting commercial event in e-commerce, concentrated around Nov 25–29
- **December**: post-Black-Friday recovery and holiday shopping

Analyzing all three months together allows us to separate structural platform behavior from seasonal distortion. Any finding that holds across all three months is structural. Any finding that diverges sharply in November is seasonal.

In [17]:
result = con.execute("""                                                                                                                                                                               
      SELECT 
          MONTH(event_time::TIMESTAMP) as month,                                                                                                                                                         
          event_type,
          COUNT(*) as events,                                                                                                                                                                            
          COUNT(DISTINCT user_id) as users,
          COUNT(DISTINCT user_session) as sessions                                                                                                                                                       
      FROM read_csv_auto('../data/raw/2019-*.csv')                                                                                                                                                       
      GROUP BY month, event_type
      ORDER BY month, event_type                                                                                                                                                                         
  """).fetchdf()

print(result) 


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   month event_type    events    users  sessions
0     10       cart    926516   337117    573097
1     10   purchase    742849   347118    629560
2     10       view  40779399  3022130   9242653
3     11       cart   3028930   826323   1743342
4     11   purchase    916939   441638    773214
5     11       view  63556110  3695598  13766768
6     12       cart   3394763   923882   1983487
7     12   purchase   1162048   500997    970005
8     12       view  62986067  4576955  15577871


### 1.3 Funnel Conversion Rates by Month

Raw counts show volume. Rates reveal behavior. We compute four rates:

| Metric | Level | Question it answers |
|--------|-------|---------------------|
| View → Cart | Event | Of all browsing clicks, what fraction lead to cart? |
| Cart → Purchase | Event | Of all cart events, what fraction convert? |
| View → Cart | User | Of all unique viewers, what fraction ever cart? |
| Cart → Purchase | User | Of unique cart-adders, what fraction purchase? |

Event-level and user-level rates diverge when power users skew the event distribution. Reporting only one is incomplete.

In [30]:
wide = pd.pivot_table(result, index='month', columns='event_type', values=['events', 'users'], aggfunc='sum')
wide.columns = ['_'.join(col) for col in wide.columns]
wide['events_cart_view_percent'] = wide['events_cart'] / wide['events_view'] * 100
wide['events_purchase_cart_percent'] = wide['events_purchase'] / wide['events_cart'] * 100
wide['users_cart_view_percent'] = wide['users_cart'] / wide['users_view'] * 100
wide['users_purchase_cart_percent'] = wide['users_purchase'] / wide['users_cart'] * 100
display(wide[['events_cart_view_percent','events_purchase_cart_percent','users_cart_view_percent','users_purchase_cart_percent']])

,events_cart_view_percent,events_purchase_cart_percent,users_cart_view_percent,users_purchase_cart_percent
month,,,,
10,2.272020,80.176597,11.154947,102.966626
11,4.765757,30.272704,22.359656,53.446171
12,5.389705,34.230608,20.185516,54.227380


---

## Finding 1 — The Two Behavioral Regimes

A single "cart abandonment rate" is not a single problem. Across the three-month dataset, two fundamentally different cart behaviors coexist — one dominated by purchase intent, one dominated by deferral. They require opposite interventions.

---

### The Numbers

| Month | Event View→Cart | Event Cart→Purchase | User View→Cart | User Cart→Purchase |
|-------|:--------------:|:-------------------:|:--------------:|:-----------------:|
| Oct   | 2.3%           | 80.2%               | 11.2%          | 103.0% *          |
| Nov   | 4.8%           | 30.3%               | 22.4%          | 53.4%             |
| Dec   | 5.4%           | 34.2%               | 20.2%          | 54.2%             |

*> 100% — investigated in Section 1.4 below.

From October to November:
- Cart rate **more than doubled** — users added to cart far more aggressively
- Cart→Purchase rate **collapsed by 62%** — most of those carts were never bought

---

### Regime 1 — Intent-Driven (dominant in October)

Users browse with real purchase intent. Adding to cart is a near-final commitment before checkout. Cart→Purchase holds above 80% at the event level. The primary leak in this regime is **browse abandonment**: users who view but never cart.

**Growth intervention:** reduce friction between discovery and cart — better recommendations, social proof, price anchoring, faster product pages.

---

### Regime 2 — Deferral-Driven (dominant in November/December)

Users treat the cart as a **wishlist and price-comparison tool**, driven by Black Friday deal-seeking psychology. Cart events spike but purchase conversion collapses to ~30%. The user is not hesitating at checkout — they are *waiting for a trigger*: a price drop, a flash sale, a stock warning.

**Growth intervention:** create the trigger — price drop alerts, stock scarcity signals, time-limited nudges.

---

### Why This Matters for Growth Teams

The two regimes coexist in every month — October is dominated by Regime 1, November/December by Regime 2. A growth team that reports a single blended cart abandonment rate is averaging two diseases into one number. The October intervention (reduce browse friction) applied to November users would be actively counterproductive — it pushes purchase pressure onto users who are in deliberate research mode. Every downstream analysis in this project segments by regime, not just by month.

---

### 1.4 The October 103% Anomaly — A Methodological Investigation

October's user-level cart→purchase rate exceeds 100%, which is mathematically impossible if all users completed their cart→purchase journey within the same calendar month. Two hypotheses:

1. **Direct purchase path**: a subset of users purchase via a Buy Now flow that bypasses the cart entirely — generating a purchase event with no preceding cart event
2. **Time-window artifact**: users who added items to cart in September (before the dataset begins) purchased in October — appearing as purchasers with no visible cart history

**Test:** identify users who purchased in any month but never generated a cart event across the entire 3-month dataset. If this population decays sharply from October → November → December, it confirms the time-window hypothesis — each new month of data makes more cart history visible, shrinking the "ghost" population.

In [41]:
con.execute("""
  WITH lifetime AS (                                                                                                                                                                                     
      SELECT                                                                                                                                                                                             
          user_id,
          MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS ever_purchased,                                                                                                                         
          MAX(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS ever_carted                                                                                                                             
      FROM read_csv_auto('../data/raw/2019-*.csv')
      GROUP BY user_id                                                                                                                                                                                   
  ),              
  direct_buyers AS (                                                                                                                                                                                     
      SELECT user_id FROM lifetime
      WHERE ever_purchased = 1 AND ever_carted = 0
  )                                                                                                                                                                                                      
  SELECT
      MONTH(event_time::TIMESTAMP) AS month,                                                                                                                                                             
      COUNT(DISTINCT e.user_id) AS direct_buyers
  FROM read_csv_auto('../data/raw/2019-*.csv') e
  INNER JOIN direct_buyers d ON e.user_id = d.user_id                                                                                                                                                    
  WHERE event_type = 'purchase'
  GROUP BY month                                                                                                                                                                                         
  ORDER BY month  


""").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,direct_buyers
0,10,81780
1,11,27926
2,12,2429


In [42]:
con.execute("""
    WITH carted AS(
        SELECT user_id, Month(event_time::TIMESTAMP) AS carted_month
        FROM read_csv_auto('../data/raw/2019-*.csv')
        WHERE event_type = 'cart'
    ),
    purchased AS(
        SELECT user_id, Month(event_time::TIMESTAMP) AS purchased_month
        FROM read_csv_auto('../data/raw/2019-*.csv')
        WHERE event_type = 'purchase'
    )
    SELECT
        c.carted_month,
        p.purchased_month,
        COUNT(DISTINCT c.user_id) AS users
        FROM carted c 
        INNER JOIN purchased p ON c.user_id = p.user_id
        GROUP BY c.carted_month, p.purchased_month
        ORDER BY c.carted_month, p.purchased_month


""").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,carted_month,purchased_month,users
0,10,10,202777
1,10,11,73703
2,10,12,60441
3,11,10,117874
4,11,11,400047
5,11,12,165561
6,12,10,97614
7,12,11,149201
8,12,12,497605


---

## Finding 2 — The Direct Buyer Anomaly Is a Time-Window Artifact

| Month | Users who purchased but never carted (any month) |
|-------|:------------------------------------------------:|
| Oct   | 81,780 |
| Nov   | 27,926 |
| Dec   | 2,429  |

The population collapses by **97% from October to December**. This pattern is the proof.

If these were genuine direct purchasers using a Buy Now flow, the behavior would be relatively stable across months — platform UX does not change that dramatically in 90 days. Instead, the sharp decline follows a clear logic: each additional month of data makes more cart history visible. By December, virtually every active user has at least one observable cart event from October or November, making it nearly impossible to appear as a "never carted" user.

**Conclusion:** the 81,780 October "direct buyers" are overwhelmingly users whose September cart events fall outside the dataset window. They purchased in October against a cart they added in September — invisible to us. The anomaly is a measurement artifact of the dataset start date, not a real platform behavior.

---

### Methodological Implication

Monthly user-level funnel rates are contaminated by cross-period user journeys. A user's cart event and their purchase event can live in different calendar months. This means:

- October's 103% is an artifact, not a real conversion rate
- The "true" monthly cart→purchase rate requires session-level analysis to be meaningful
- All funnel metrics in this project are interpreted with this limitation in mind

The session-level funnel — built next — is the clean unit of measurement that avoids this contamination entirely.